In [5]:
from IPython.display import Image

In [4]:
Image(url='https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3-VL/qwen3vl_arc.jpg', width=500)

### visual tokens

$$
N_{tokens} = \left( \frac{H_{resized}}{32} \times \frac{W_{resized}}{32} \right) + N_{special}
$$

- $H_{resized}, W_{resized}$：是经过动态分辨率调整后，被补齐为 32 倍数的图像高度和宽度。
- $32$：是 Qwen3-VL 的有效 Token 步长（Effective Stride）。
    - 技术细节：Qwen3-VL 使用的 Patch Size 为 16（Qwen2.5-VL 是 14），配合 2x2 的池化（Pooling），最终每个 Token 对应原图上的 16 × 2 = 32 像素区域。
    - 第1层（ViT）： 把图切成小块（16x16像素），每块生成一个“特征向量”。
    - 第2层（Adapter）： 把相邻的 4 个（2x2）特征向量“捏”在一起，合并成 1 个更浓缩的向量。
        - 模型取出 $2 \times 2$ 的特征向量网格（即左上、右上、左下、右下 4 个向量），将它们合并（池化）成 1 个 向量。
    - 结果： LLM 最终看到的 1 个 Token，实际上包含了原始图片中 $2 \times 2$ 个小块的信息。
- $N_{special}$：特殊标记 Token，通常为 2 个（即 `<|vision_start|>` 和 `<|vision_end|>`）。

### Interleaved MROPE

In [7]:
Image(url='./imgs/m-rope.png', width=800)

$$
m\rightarrow (t, h, w)
$$

> 一个 visual token 的位置不再只是一个顺序递增的数字，而是 (t, h, d)，即是一个 3d position：第几帧，第几行，第几列

$$
\theta_i = b^{- \frac{2i}{d}}, \quad \mathbf{R}_{\Theta, m}^i \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}
$$

- 在 Qwen2.5-VL 中，MROPE 采用的是**分块（Chunking）**策略。
    - 它把特征向量的维度切成三块，分别分配给 $t$、$h$ 和 $w$ 。
    - 假设向量维度 $D=96$，它可能把前 32 维给时间 $t$，中间 32 维给高度 $h$，最后 32 维给宽度 $w$。
        - 结构示意：[ t, t, t ... | h, h, h ... | w, w, w ... ]
- RoPE 的频率特性（关键点）： 在旋转位置编码（RoPE）的数学原理中，向量的不同维度对应着不同的旋转频率。
    - 低维度（靠前的索引）：对应高频分量，擅长捕捉局部、细节信息。
    - 高维度（靠后的索引）：对应低频分量，擅长捕捉长距离、全局依赖 。
- 由于采用了“分块”设计，$t$、$h$、$w$ 被固定在向量的不同区段，导致它们被迫绑定了不同的频率。
    - 如果 $t$ 被分在靠前的维度，它就只能学到高频信息（短时变化），而学不到长周期的低频信息（长视频的上下文）。
    - 这就是文中提到的“频谱不平衡（Imbalanced Frequency Spectrum）”，它直接扼杀了模型理解长视频的能力 。
- Qwen3-VL 的改进方案非常直观：打散重排。
    - 不再把 $t, h, w$ 分块隔离，而是将它们**交错（Interleave）**排列在整个向量维度中 。
    - 结构示意：[ t, h, w, t, h, w, t, h, w ... ]

$$
\text{Pos}(i) =
\begin{cases}
t, & 0 \le i < D/6 \quad (\text{对应高频段}) \\
h, & D/6 \le i < D/3 \quad (\text{对应中频段}) \\
w, & D/3 \le i < D/2 \quad (\text{对应低频段})
\end{cases}
$$
- 这种切分将时间维度（Temporal）完全限制在高频通道 (High-frequency channels)，直接导致模型难以捕捉长距离的时间依赖（Long-range video modeling），

在标准 RoPE 中，每个维度对 $(2i, 2i+1)$ 对应一个频率 $\theta_i$。在 Interleaved MROPE 中，为了确保 $t, h, w$ 都能共享整个频谱（从高频到低频），频率 $\theta_i$ 是按“组”共享的。即每 3 个维度对（一个 $t$，一个 $h$，一个 $w$）使用同一个基准频率。公式如下：

$$\theta_i = 10000^{- \frac{2 \lfloor i / 3 \rfloor}{d_{\text{sub}}}}=10000^{- \frac{2 \lfloor i / 3 \rfloor}{d_{\text{head}}/3}}$$

其中：
- $i$ 是维度对的索引，范围从 $0$ 到 $d_{\text{head}}/2 - 1$。
    - rope 是2d旋转，总对数 $d_{\text{head}}/2$
- $\lfloor i / 3 \rfloor$ 表示向下取整，意味着索引为 $0, 1, 2$ 的三个维度对共享同一个频率指数 $0$；索引 $3, 4, 5$ 共享指数 $1$，以此类推。
- $d_{\text{sub}}$ 是每个模态分到的有效维度数，通常为 $d_{\text{head}} / 3$。这保证了频率范围能完整覆盖 $(0, 1)$ 区间，实现“全频段覆盖”。
    - 标准 RoPE 的设计目标是让频率指数的范围覆盖 $[-1, 0]$，对应的频率范围是 $[10000^{-1}, 10000^0]$。

旋转矩阵 $\mathcal{R}_{Interleaved}$Interleaved MROPE 的旋转矩阵是一个块对角矩阵，其特殊之处在于位置索引 $m$ 的取值在 $t, h, w$ 之间轮转。矩阵形式如下：

$$
\mathbf{R}_{Interleaved} = \begin{pmatrix}
\mathbf{R}_0 & 0 & 0 & \cdots \\
0 & \mathbf{R}_1 & 0 & \cdots \\
0 & 0 & \mathbf{R}_2 & \cdots \\
\vdots & \vdots & \vdots & \ddots
\end{pmatrix}
$$

其中每一个 $2 \times 2$ 的子块 $\mathbf{R}_i$ 定义为：$$\mathbf{R}_i = \begin{pmatrix} 
\cos(m_i \theta_i) & -\sin(m_i \theta_i) \\ 
\sin(m_i \theta_i) & \cos(m_i \theta_i) 
\end{pmatrix}$$关键在于 $m_i$ 的交错选择逻辑：$$m_i = 
\begin{cases} 
t & \text{if } i \pmod 3 = 0 \quad (\text{时间/帧索引}) \\
h & \text{if } i \pmod 3 = 1 \quad (\text{高度索引}) \\
w & \text{if } i \pmod 3 = 2 \quad (\text{宽度索引})
\end{cases}$$

- 这种 "Round-robin"（轮询）分布确保了每个轴（$t, h, w$）都能均匀地采样到从高频到低频的完整频谱。

| 维度对索引 $i$ | 分配的频率 $\theta_i$ | 旋转依据的位置 $m_i$ | 频率特性 |
| :--- | :--- | :--- | :--- |
| **0** | $\theta_{\text{high}}$ | **$t$ (时间)** | 高频（捕捉细节） |
| **1** | $\theta_{\text{high}}$ | **$h$ (高度)** | 高频（捕捉细节） |
| **2** | $\theta_{\text{high}}$ | **$w$ (宽度)** | 高频（捕捉细节） |
| **3** | $\theta_{\text{mid}}$ | **$t$ (时间)** | 中频 |
| **4** | $\theta_{\text{mid}}$ | **$h$ (高度)** | 中频 |
| **5** | $\theta_{\text{mid}}$ | **$w$ (宽度)** | 中频 |
| ... | ... | ... | ... |
| **Max** | $\theta_{\text{low}}$ | **$w$ (宽度)** | 低频（捕捉全局） |